# Scenarios and stress testing

**Prerequisites:** complete curriculum notebooks **03** (market data and curves), **05** (pricing fundamentals), and **11** (portfolio construction and valuation) before this notebook. You should be comfortable with `MarketContext` JSON, instrument/portfolio specs as JSON, and the idea of repricing against a market snapshot.


## Concepts: scenarios and stress testing

A **scenario** is a declarative set of **operations** (curve shocks, spread moves, statement overrides, and similar) packaged as a **`ScenarioSpec`** (JSON). **Stress testing** means applying those operations to a market (and optionally a financial model), then **revaluing** positions or a portfolio so you can compare base versus stressed PVs and P&L.

finstack separates **authoring** (templates, `build_scenario_spec`, `compose_scenarios`, validation) from **application** (`apply_scenario_to_market`, `apply_scenario`) and from **portfolio workflows** (`apply_scenario_and_revalue`). Keeping scenarios as JSON makes them easy to log, diff, and compose.


### Imports and built-in templates

`list_builtin_templates` returns template IDs from the embedded registry. Use `list_builtin_template_metadata` when you need descriptions in bulk (JSON string).


In [1]:
import json

from finstack.scenarios import (
    apply_scenario,
    apply_scenario_to_market,
    build_from_template,
    build_scenario_spec,
    build_template_component,
    compose_scenarios,
    list_builtin_template_metadata,
    list_builtin_templates,
    list_template_components,
    parse_scenario_spec,
    validate_scenario_spec,
)

templates = list_builtin_templates()
print("Available templates:")
for t in templates:
    print(f"  {t}")
print(f"Total: {len(templates)}")
meta = list_builtin_template_metadata()
print("Template metadata JSON length:", len(meta), "chars")


Available templates:
  gfc_2008
  covid_2020
  rate_shock_2022
  svb_2023
  ltcm_1998
Total: 5
Template metadata JSON length: 2539 chars


### Build from template

`build_from_template` materializes a full `ScenarioSpec` JSON string. Composite templates expose **components** via `list_template_components` and `build_template_component`.

Historical templates bundle rates, credit, equity, vol, and FX. For a **toy market** (only discount curves), we build a **rates component** spec for `apply_*` calls so operations reference curves that exist (`USD-SOFR` here, alongside `USD-OIS`).


In [2]:
spec_json = None
scenario_for_apply = None
if templates:
    tid = templates[0]
    spec_json = build_from_template(tid)
    print(f"Built from {tid!r} (first 500 chars):")
    print(spec_json[:500])
    components = list_template_components(tid)
    print(f"Components: {components}")
    rates_ids = [c for c in components if "rates" in c.lower()]
    if rates_ids:
        scenario_for_apply = build_template_component(tid, rates_ids[0])
        print(f"scenario_for_apply <- component {rates_ids[0]!r} (first 220 chars):")
        print(scenario_for_apply[:220] + "...")
    elif components:
        scenario_for_apply = build_template_component(tid, components[0])
        print(f"scenario_for_apply <- first component {components[0]!r}")
        print(scenario_for_apply[:220] + "...")
    else:
        scenario_for_apply = spec_json
        print("No components; using full template for apply (needs full market).")
else:
    print("No built-in templates in this build — skip template-driven cells below.")


Built from 'gfc_2008' (first 500 chars):
{"id":"gfc_2008","name":"Global Financial Crisis 2008","description":"Lehman collapse - rates rally, credit blowout, equity crash, vol spike","operations":[{"kind":"curve_parallel_bp","curve_kind":"discount","curve_id":"USD-SOFR","bp":-200.0},{"kind":"curve_node_bp","curve_kind":"discount","curve_id":"USD-SOFR","nodes":[["2Y",-50.0],["5Y",0.0],["10Y",50.0],["30Y",75.0]],"match_mode":"interpolate"},{"kind":"curve_parallel_bp","curve_kind":"par_cds","curve_id":"USD-IG","bp":300.0},{"kind":"curve_p
Components: ['gfc_2008_rates', 'gfc_2008_credit', 'gfc_2008_equity', 'gfc_2008_vol', 'gfc_2008_fx']
scenario_for_apply <- component 'gfc_2008_rates' (first 220 chars):
{"id":"gfc_2008_rates","name":"GFC 2008 - Rates","description":null,"operations":[{"kind":"curve_parallel_bp","curve_kind":"discount","curve_id":"USD-SOFR","bp":-200.0},{"kind":"curve_node_bp","curve_kind":"discount","cu...


### Custom scenario specs

Use `build_scenario_spec` with a JSON list of operations. Each operation is a JSON object with a **`kind`** tag (`curve_parallel_bp`, `equity_price_pct`, ...) matching the Rust serde schema. `validate_scenario_spec` returns `True` when the payload parses cleanly.


In [3]:
ops = json.dumps(
    [
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "bp": 100.0,
        }
    ]
)
custom = build_scenario_spec(
    "rate-shock-100",
    ops,
    "Rate Shock +100bp",
    "Parallel shift of USD-OIS by 100bp",
)
print(custom)
valid = validate_scenario_spec(custom)
print(f"Valid: {valid}")
parsed = parse_scenario_spec(custom)
print("parse_scenario_spec length:", len(parsed), "chars")


{"id":"rate-shock-100","name":"Rate Shock +100bp","description":"Parallel shift of USD-OIS by 100bp","operations":[{"kind":"curve_parallel_bp","curve_kind":"discount","curve_id":"USD-OIS","bp":100.0}],"priority":0,"resolution_mode":"most_specific_wins"}
Valid: True
parse_scenario_spec length: 253 chars


### Compose scenarios

`compose_scenarios` takes a JSON **list** of scenario specs and returns a single merged spec string.


In [4]:
if len(templates) >= 2:
    s1 = build_from_template(templates[0])
    s2 = build_from_template(templates[1])
    composed = compose_scenarios(json.dumps([json.loads(s1), json.loads(s2)]))
    print("Composed scenario (first 300 chars):", composed[:300])
else:
    print("Need at least 2 built-in templates to demonstrate compose_scenarios (have", len(templates), ").")


Composed scenario (first 300 chars): {"id":"gfc_2008+covid_2020","name":"Global Financial Crisis 2008 + COVID 2020 Market Shock","description":null,"operations":[{"kind":"curve_parallel_bp","curve_kind":"discount","curve_id":"USD-SOFR","bp":-200.0},{"kind":"curve_node_bp","curve_kind":"discount","curve_id":"USD-SOFR","nodes":[["2Y",-50


### Apply scenario to market

`apply_scenario_to_market` returns a dict with `market_json`, `operations_applied`, and `warnings`. The stressed curve set is in `market_json`.


**JSON shape vs content.** Scenario binding (`apply_scenario_to_market` / `apply_scenario`) returns **`market_json` as a compact string** (single-line JSON). A notebook that prints `MarketContext.to_json()` often uses **`json.dumps(..., indent=2)`** for readability. The **curve set is the same logical data** once parsed—only whitespace and key ordering may differ.

**Extrapolation warnings.** If a scenario references **tenor pillars beyond** the curve’s built range (for example **10Y** and **30Y** nodes on a **5Y**-long curve), the engine applies the curve’s **extrapolation policy** (often **flat forward** off the last pillar). The resulting bumps can be **milder or distorted** versus a full-term structure—treat **`warnings`** as a **sanity check** that the stress is applied where you think it is.

In [5]:
from datetime import date

from finstack.core.market_data import DiscountCurve, MarketContext

knots = [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (5.0, 0.75)]
as_of_d = date(2025, 1, 15)
mc = MarketContext()
mc.insert(DiscountCurve("USD-OIS", as_of_d, knots, day_count="act_365f"))
mc.insert(DiscountCurve("USD-SOFR", as_of_d, knots, day_count="act_365f"))
market_json = mc.to_json()
print("Base market_json length:", len(market_json))

base_doc = json.loads(market_json)
base_curve_ids = sorted(
    c.get("id") for c in base_doc.get("curves", []) if isinstance(c, dict) and c.get("id")
)
pretty_snapshot = json.dumps(base_doc, indent=2)
pretty_ids = sorted(
    c.get("id") for c in json.loads(pretty_snapshot).get("curves", []) if isinstance(c, dict) and c.get("id")
)
print("Curve IDs: base parse == pretty parse:", base_curve_ids == pretty_ids, base_curve_ids)

if templates and scenario_for_apply:
    result = apply_scenario_to_market(scenario_for_apply, market_json, "2025-01-15")
    print(f"Operations applied: {result['operations_applied']}")
    print(f"Warnings: {result['warnings']}")
    if result["warnings"]:
        print(
            "(Sanity check) Long-tenor node shocks on a short curve trigger extrapolation;"
            " compare DF behavior at the last pillar vs far tenors."
        )
    stressed_market = result["market_json"]
    print("Stressed market_json length:", len(stressed_market))
    stressed_ids = sorted(
        c.get("id")
        for c in json.loads(stressed_market).get("curves", [])
        if isinstance(c, dict) and c.get("id")
    )
    print("Curve IDs: base vs stressed (same ids, different formatting):", base_curve_ids == stressed_ids)
    print("  base:", base_curve_ids, "| stressed:", stressed_ids)
else:
    print("Skipping apply_scenario_to_market (no template or scenario_for_apply).")


Base market_json length: 1292
Operations applied: 2
Warnings: ["Tenor '10Y' (10.01Y) extrapolates beyond curve max (5.00Y) by 5.00Y. Using flat extrapolation to last pillar.", "Tenor '30Y' (30.02Y) extrapolates beyond curve max (5.00Y) by 25.02Y. Using flat extrapolation to last pillar."]
Stressed market_json length: 792


### Apply to market and statements model

`apply_scenario` mutates both **market** and **model** JSON when operations target each side. The return dict includes updated `market_json` and `model_json`.


In [6]:
from finstack.statements import ModelBuilder

b = ModelBuilder("stress-model")
b.periods("2025Q1..Q2", None)
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0)])
b.compute("gross_profit", "revenue - cogs")
model_json = b.build().to_json()
print("Model JSON length:", len(model_json))

if templates and scenario_for_apply:
    full_result = apply_scenario(scenario_for_apply, market_json, model_json, "2025-01-15")
    print(f"Operations applied: {full_result['operations_applied']}")
    print(f"Warnings: {full_result.get('warnings', [])}")
    print("Returned keys:", sorted(full_result.keys()))
else:
    print("Skipping apply_scenario (no template or scenario_for_apply).")


Model JSON length: 865
Operations applied: 2
Warnings: ["Tenor '10Y' (10.01Y) extrapolates beyond curve max (5.00Y) by 5.00Y. Using flat extrapolation to last pillar.", "Tenor '30Y' (30.02Y) extrapolates beyond curve max (5.00Y) by 25.02Y. Using flat extrapolation to last pillar."]
Returned keys: ['market_json', 'model_json', 'operations_applied', 'warnings']


## Mini-example: compose stress, revalue portfolio

Build a tiny **portfolio** JSON with one USD deposit discounted off **`USD-SOFR`** (the id shocked by built-in rates components), value it against the **base** market, then apply the **rates component** scenario and revalue via `apply_scenario_and_revalue`. Compare totals with `portfolio_result_total_value`.


In [7]:
from finstack.portfolio import apply_scenario_and_revalue, value_portfolio

def total_base_from_valuation(val_json: str) -> float:
    """`value_portfolio` returns `PortfolioValuation` JSON; read base-currency total."""
    d = json.loads(val_json)
    return float(d["total_base_ccy"]["amount"])

portfolio = json.dumps(
    {
        "id": "test-port",
        "as_of": "2025-01-15",
        "base_ccy": "USD",
        "entities": {"E1": {"id": "E1"}},
        "positions": [
            {
                "position_id": "P1",
                "entity_id": "E1",
                "instrument_id": "DEP-1",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-1",
                        "notional": {"amount": 1000000.0, "currency": "USD"},
                        "start_date": "2025-01-15",
                        "maturity": "2025-07-15",
                        "day_count": "Act360",
                        "quote_rate": 0.05,
                        "discount_curve_id": "USD-SOFR",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            }
        ],
    }
)

base_val = value_portfolio(portfolio, market_json)
base_total = total_base_from_valuation(base_val)
print(f"Base portfolio total: {base_total:,.2f}")

if templates and scenario_for_apply:
    stressed_val, report = apply_scenario_and_revalue(
        portfolio, scenario_for_apply, market_json
    )
    stressed_total = total_base_from_valuation(stressed_val)
    dpl = stressed_total - base_total
    print(f"Stressed total:       {stressed_total:,.2f}")
    print(f"P&L impact:           {dpl:,.2f}")
    print(
        "(Sanity check) Large |P&L| here is expected when parallel + key-rate shocks hit a short SOFR curve"
        " with extrapolation; deposit PV sign follows cash-flow convention (see portfolio construction notebook)."
    )
    rep_obj = json.loads(report)
    if rep_obj.get("warnings"):
        print("(Note) Scenario warnings (e.g. extrapolation) also apply to this repricing path:", rep_obj["warnings"])
    print("Scenario report (first 400 chars):", report[:400])
else:
    print("Skipping stressed path (no template or scenario_for_apply).")


Base portfolio total: -278.89
Stressed total:       9,121.95
P&L impact:           9,400.84
Scenario report (first 400 chars): {"operations_applied":2,"warnings":["Tenor '10Y' (10.01Y) extrapolates beyond curve max (5.00Y) by 5.00Y. Using flat extrapolation to last pillar.","Tenor '30Y' (30.02Y) extrapolates beyond curve max (5.00Y) by 25.02Y. Using flat extrapolation to last pillar."],"rounding_context":"Bankers"}


## Takeaways

- **Templates** (`list_builtin_templates`, `build_from_template`) are the fastest way to obtain realistic scenario JSON; **components** let you inspect or rebuild pieces of composite templates.
- **Custom specs** use `build_scenario_spec` + a JSON operation list; **`validate_scenario_spec`** / **`parse_scenario_spec`** keep payloads canonical.
- **`compose_scenarios`** merges multiple specs for stacked stresses (order follows engine rules and priorities).
- **`apply_scenario_to_market`** and **`apply_scenario`** produce updated JSON snapshots for downstream pricing or statements.
- **`value_portfolio`** returns **`PortfolioValuation` JSON** (use `total_base_ccy` for the base-currency total); **`portfolio_result_total_value`** expects the richer **`PortfolioResult`** wrapper from other flows.
- **`apply_scenario_and_revalue`** returns stressed **`PortfolioValuation` JSON** plus a report string — compare base vs stressed totals for P&L impact.

**Next:** combine scenario sets with your own reporting (limits, dashboards) using the same JSON spec strings.
